## Exploring the ODISSEI Knowledge Graph via SPARQL

Goal: extend the original query to also retrieve the **column name** and **description** of the tijdstip variable.
First, we explore what properties exist on variable and dataset nodes.

In [3]:
from pathlib import Path
import polars as pl
from life_event_metadata.utils import get_root
from life_event_metadata.utils import run_query
from life_event_metadata.utils import wide_print

In [4]:
ROOT = get_root()

### Step 1: What properties does a variable node have?

Pick one dataset (ADOPTIEKINDEREN) and dump all predicates + values on its variables.

In [5]:
q_variable_props = """
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>
prefix dct: <http://purl.org/dc/terms/>

SELECT ?var ?p ?o WHERE {
  ?dataset dct:alternative "ADOPTIEKINDEREN" .
  ?dataset vi:odisseiVariable ?var .
  ?var ?p ?o .
}
LIMIT 200
"""

var_props = run_query(q_variable_props)
wide_print(var_props)

shape: (200, 3)
┌──────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ var                                                                      ┆ p                                                                                     ┆ o                                                                                                           │
│ ---                                                                      ┆ ---                                                                                   ┆ ---                                                                                                         │
│ str                                                                      ┆ str                                                                               

### Step 2: What properties does a dataset node have?

Same idea: dump all predicates on the dataset node itself, to find the full dataset name.

In [ ]:
q_dataset_props = """
prefix dct: <http://purl.org/dc/terms/>

SELECT ?p ?o WHERE {
  ?dataset dct:alternative "ADOPTIEKINDEREN" .
  ?dataset ?p ?o .
}
LIMIT 200
"""

ds_props = run_query(q_dataset_props)
wide_print(ds_props)

### Step 3: Inspect the distinct predicates

Summarise the predicates we found to identify which ones hold the column name and description.

In [ ]:
print("Variable predicates:")
wide_print(var_props["p"].unique().sort())
print()
print("Dataset predicates:")
wide_print(ds_props["p"].unique().sort())

### Step 3b: What does each variable predicate look like for one concrete variable?

`odisseiConceptVariableName` gives a human-readable name, not a raw column ID. Let's check what `odisseiConceptVariableID` looks like — that might be the actual column name in the dataset.

### Step 4: Extend the original query

The variable node has two levels of properties:
- **Concept level** (`odisseiConceptVariable*`): abstract, shared across datasets
- **Variable level** (`odisseiVariable*`): concrete, specific to a dataset version

For the column name we want `vi:odisseiVariableName` (e.g., `ADODatumV`), not the concept name.
For the description we use `vi:odisseiVariableDefinition`.

In [ ]:
q_extended = """
prefix schema: <http://schema.org/>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>
prefix skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?datasetTitle ?altTitle ?tijdstipColName ?tijdstipVarLabel ?tijdstipDescription WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset a schema:Dataset .
  ?dataset dct:alternative ?altTitle .
  ?dataset dct:title ?datasetTitle .

  # Person identifier variable
  ?dataset vi:odisseiVariable ?var1 .
  ?var1 vi:odisseiVariableVocabularyURI ?var1VocabURI .
  FILTER(?var1VocabURI = <https://w3id.org/odissei/cv/cbs/variableThesaurus/c650ad27af3f9e2b081c2b2f3698ae9eeb2a502f919b2f220d9f5b29afd6f337e>)

  # Tijdstip variable
  ?dataset vi:odisseiVariable ?tijdstipVar .
  ?tijdstipVar ?p ?varLabel .
  FILTER(isLiteral(?varLabel) && CONTAINS(LCASE(STR(?varLabel)), LCASE("Tijdstip")))
  ?tijdstipVar vi:odisseiVariableVocabularyURI ?tijdstipVocabURI .
  ?tijdstipVocabURI skos:prefLabel ?tijdstipVarLabel .

  # Column name (actual column in the dataset, e.g. ADODatumV)
  ?tijdstipVar vi:odisseiVariableName ?tijdstipColName .

  # Description of the tijdstip variable
  ?tijdstipVar vi:odisseiVariableDefinition ?tijdstipDescription .
}
ORDER BY ?altTitle
"""

extended = run_query(q_extended)
wide_print(extended.head(10))

In [ ]:
extended.head()

In [ ]:
extended.shape

In [ ]:
extended.write_csv(Path(ROOT, "data/data-designs-extended.csv"))

### Exploring `extended`

In [ ]:
extended = pl.read_csv(Path(ROOT, "data/data-designs-extended.csv"))

In [ ]:
counts_by_dataset = extended.group_by(["altTitle", "datasetTitle"]).len().rename({"len": "n"})

In [ ]:
counts_by_dataset.sort("n", descending=True).head(10)
# this is not super informative actually. this simply says how many time columns there are in a dataset.
# it's not obvious that "more is better"

In [ ]:
wide_print(extended.filter(pl.col("altTitle") == "NZREPISODETAB")["tijdstipColName", "tijdstipVarLabel"], -1)

### Notes
- Verified column name in one case, `ADOJrV`: https://portal.odissei.nl/dataset.xhtml?persistentId=doi:10.57934/0B01E410802168E1
- There's many more variables - which ones do we need? 
- why do we have now "only" 579? is this good or bad? b/c of missingness of the col names?
- See datum notatie in the `tijdstipDescription` column - is this the single source of this information or can it also be buried elsewhere?
- some next filtering steps:
   - exclude events that do not vary over time, or decide if they should be added to background dictionary
- Both `TAB` and `BUS` are available??
- SPOLISBUS - which date variable matters?

### Additional metadata to query

#### At the dataset level
- publication date? Does this correspond to the first measurement, and are all data sets continued until today?
   - it seems so for Burennetwerk for instance
- CBS specific metadata
   - "valid from", "valid until"
- Frequency - this is a property of the dataset and thus all variables associated with the dataset
  - note: SPOLISBUS has frequency "Niet eenduidig"
- "Sampling procedure" has text and *may* contain information about missingness (see INPATAB)

#### At the variable level (at least for the Notes)
- data type
- label
- definition
- "Conceptual Variable Valid from" - this is not the same as dataset validity (can go back further than dataset validity). what does it mean? does it mean the same variable can be found in other datasets?

### How to select events?
- "those with the most signal"
- number of potential attributes? variability of attributes? frequency?
- do we need an automatic pre-processing layer? what did Tanzir do here?


In [ ]:
## Update 16/03/26

### Frequency of data collection

First, discover all predicates containing "frequency" to confirm the right namespace.

In [ ]:
q_frequency_modalities = """
prefix ss: <https://portal.odissei.nl/schema/socialscience#>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>

SELECT ?frequency (COUNT(DISTINCT ?dataset) AS ?count) WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset ss:frequencyOfDataCollection ?frequency .

  # Person identifier variable
  ?dataset vi:odisseiVariable ?var1 .
  ?var1 vi:odisseiVariableVocabularyURI ?var1VocabURI .
  FILTER(?var1VocabURI = <https://w3id.org/odissei/cv/cbs/variableThesaurus/c650ad27af3f9e2b081c2b2f3698ae9eeb2a502f919b2f220d9f5b29afd6f337e>)
}
GROUP BY ?frequency
ORDER BY DESC(?count)
"""

freq_modalities = run_query(q_frequency_modalities)
wide_print(freq_modalities)

In [ ]:
freq_modalities = freq_modalities.with_columns(pl.col("count").cast(pl.Int64)).sort("count", descending=True)

In [ ]:
freq_modalities.head(10)

### CBS datasets filtered by frequency

In [ ]:
selected_frequencies = [
    "Jaar",
    "Kalenderjaar",
    "Niet eenduidig",
    "Stand",
    "Maand",
]

### Most common keywords across CBS datasets

In [ ]:
q_top_keywords = """
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>

SELECT ?keyword (COUNT(DISTINCT ?dataset) AS ?count) WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset citation:keyword ?kwNode .
  ?kwNode citation:keywordValue ?keyword .
}
GROUP BY ?keyword
ORDER BY DESC(?count)
LIMIT 20
"""

top_keywords = run_query(q_top_keywords)
wide_print(top_keywords.with_columns(pl.col("count").cast(pl.Int64)), -1)

In [ ]:
values_clause = " ".join(f'"{f}"' for f in selected_frequencies)

q_datasets_by_freq = f"""
prefix ss: <https://portal.odissei.nl/schema/socialscience#>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix schema: <http://schema.org/>

SELECT DISTINCT ?altTitle ?datasetTitle ?frequency WHERE {{
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset a schema:Dataset .
  ?dataset dct:alternative ?altTitle .
  ?dataset dct:title ?datasetTitle .
  ?dataset ss:frequencyOfDataCollection ?frequency .
  VALUES ?frequency {{ {values_clause} }}
}}
ORDER BY ?frequency ?altTitle
"""

datasets_by_freq = run_query(q_datasets_by_freq)
print(f"{datasets_by_freq.shape[0]} datasets")

In [ ]:
datasets_by_freq.head(10)

In [ ]:
values_clause = " ".join(f'"{f}"' for f in selected_frequencies)

q_datasets_by_freq = f"""
prefix ss: <https://portal.odissei.nl/schema/socialscience#>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix schema: <http://schema.org/>
prefix vi: <https://portal.odissei.nl/schema/variableInformation#>

SELECT DISTINCT ?altTitle ?datasetTitle ?frequency WHERE {{
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset a schema:Dataset .
  ?dataset dct:alternative ?altTitle .
  ?dataset dct:title ?datasetTitle .
  ?dataset ss:frequencyOfDataCollection ?frequency .
  VALUES ?frequency {{ {values_clause} }}

  # Person identifier variable
  ?dataset vi:odisseiVariable ?var1 .
  ?var1 vi:odisseiVariableVocabularyURI ?var1VocabURI .
  FILTER(?var1VocabURI = <https://w3id.org/odissei/cv/cbs/variableThesaurus/c650ad27af3f9e2b081c2b2f3698ae9eeb2a502f919b2f220d9f5b29afd6f337e>)
}}
ORDER BY ?frequency ?altTitle
"""

datasets_by_freq = run_query(q_datasets_by_freq)
print(f"{datasets_by_freq.shape[0]} datasets")

In [ ]:
extended.shape, datasets_by_freq.shape

#### Datasets without tijdstip column

In [ ]:
datasets_by_freq.join(extended, on="altTitle", how="inner").select(pl.col("altTitle")).unique().shape

In [ ]:
subset = datasets_by_freq.join(extended, on="altTitle", how="anti")

In [ ]:
subset.shape

In [ ]:
subset.filter(pl.col("altTitle").str.contains("HOOGS|INPA|NCO"))

In [ ]:
wide_print(subset, 20)

In [ ]:
subset.write_csv(Path(ROOT, "data/data-designs-without-tijdstip-with-frequency.csv"))

In [ ]:
extended.head()

In [ ]:
subset.head()

In [ ]:
full_df = pl.concat(
    [
        extended.select(pl.col("altTitle")),
        subset.select(pl.col("altTitle")),
    ],
)
full_df.head()

In [ ]:
full_df.shape

### ELSST concepts

In [ ]:
q_keyword_predicates = """
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>

SELECT DISTINCT ?p WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset ?p ?o .
  FILTER(CONTAINS(LCASE(STR(?p)), "keyword") || CONTAINS(LCASE(STR(?p)), "elsst"))
}
"""

kw_preds = run_query(q_keyword_predicates)
wide_print(kw_preds, -1)

In [ ]:
# Now explore the structure of the ELSST enrichment nodes
q_elsst_structure = """
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>
prefix enrich: <https://portal.odissei.nl/schema/enrichments#>

SELECT DISTINCT ?p ?o WHERE {
  ?creator citation:authorName 'Centraal Bureau voor Statistiek' .
  ?dataset dct:creator ?creator .
  ?dataset enrich:enrichedElsstClassification ?elsstNode .
  ?elsstNode ?p ?o .
}
LIMIT 50
"""

elsst_structure = run_query(q_elsst_structure)
wide_print(elsst_structure, -1)

In [ ]:
q = """
prefix ss: <https://portal.odissei.nl/schema/socialscience#>
prefix dct: <http://purl.org/dc/terms/>
prefix citation: <https://dataverse.org/schema/citation/>

SELECT ?frequency (COUNT(DISTINCT ?dataset) AS ?count) WHERE {
  VALUES ?authorName { 'Centraal Bureau voor Statistiek' 'Centraal Bureau voor de Statistiek (CBS)' }
  ?dataset dct:creator ?creator .
  ?creator citation:authorName ?authorName .
  ?dataset ss:frequencyOfDataCollection ?frequency .
}
GROUP BY ?frequency
ORDER BY DESC(?count)
"""

In [ ]:
res = run_query(q)

In [ ]:
wide_print(res, -1)